# Road Following Live (Real JetRacer Hardware)

This notebook executes the trained Road Following model autonomously on the physical **JetRacer** car using real-time camera feed and **Stanley / PID Control**.

### 1. Setup Environment & Load Model

In [ ]:
import os
import sys
from pathlib import Path
import torch
import torchvision

# Add parent directory to sys.path to access PID.py and utils.py
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

from utils import preprocess
from PID import PIDController, StanleyController

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model architecture (ResNet-18 or TensorRT model if available)
model_path = 'road_following_model.pth'
model_trt_path = 'model_08-06_trt.pth'

model = None
if os.path.exists(model_trt_path):
    try:
        from torch2trt import TRTModule
        model = TRTModule()
        model.load_state_dict(torch.load(model_trt_path))
        print(f"Loaded TensorRT optimized model from '{model_trt_path}'")
    except Exception as e:
        print(f"Could not load TensorRT model: {e}, falling back to PyTorch model.")

if model is None:
    model = torchvision.models.resnet18(pretrained=False)
    model.fc = torch.nn.Linear(512, 2)
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded PyTorch model weights from '{model_path}'")
    else:
        print(f"Warning: '{model_path}' not found! Please train a model first in interactive_regression.ipynb")
    model = model.to(device).eval()


### 2. Initialize JetRacer Hardware (`NvidiaRacecar` & `CSICamera`)

In [ ]:
from jetracer.nvidia_racecar import NvidiaRacecar
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

car = NvidiaRacecar()
camera = CSICamera(width=224, height=224, capture_fps=65)
camera.running = True
print("JetRacer hardware and CSI Camera initialized successfully!")


### 3. Basic Controller (P-Gain + Bias)

In [ ]:
import cv2
import ipywidgets
import traitlets
import threading
import time
from IPython.display import display
from ipywidgets import Layout

slider_style = {'description_width': '140px'}

# Control Sliders with numeric readout display enabled
network_output_slider = ipywidgets.FloatSlider(description='Network Output', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=slider_style)
steering_gain_slider  = ipywidgets.FloatSlider(description='Steering Gain', min=-2.0, max=2.0, value=-0.7, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
steering_bias_slider  = ipywidgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)
steering_value_slider = ipywidgets.FloatSlider(description='Final Steering', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=slider_style)
throttle_slider       = ipywidgets.FloatSlider(description='Throttle', min=-1.0, max=1.0, value=0.15, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=slider_style)

# Live Stream & Prediction Preview Widget
state_widget = ipywidgets.ToggleButtons(options=['Off', 'Autonomous Live'], description='Mode', value='Off')
prediction_widget = ipywidgets.Image(value=bgr8_to_jpeg(camera.value), format='jpeg', width=camera.width, height=camera.height)

live_active = False

def live_drive_loop():
    global live_active
    while live_active:
        try:
            image = camera.value
            preprocessed = preprocess(image)
            
            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()
            
            x = float(output[0])
            y = float(output[1]) if len(output) > 1 else 0.0
            
            network_output_slider.value = x
            
            # Calculate final steering value with Gain & Bias
            steering = x * steering_gain_slider.value + steering_bias_slider.value
            steering = max(-1.0, min(1.0, steering))
            steering_value_slider.value = steering
            
            # Update real car hardware steering and throttle
            car.steering = steering
            car.throttle = throttle_slider.value
            
            # Draw predicted target dot (blue circle) on video preview
            px = int(camera.width * (x / 2.0 + 0.5))
            py = int(camera.height * (y / 2.0 + 0.5))
            
            prediction = image.copy()
            prediction = cv2.circle(prediction, (px, py), 8, (255, 0, 0), 3)
            prediction_widget.value = bgr8_to_jpeg(prediction)
            
        except Exception as e:
            print(f"Error in live drive loop: {e}")
            break
            
        time.sleep(0.02)  # ~50 FPS loop

def on_state_change(change):
    global live_active
    if change['new'] == 'Autonomous Live':
        if not live_active:
            live_active = True
            t = threading.Thread(target=live_drive_loop, daemon=True)
            t.start()
    else:
        live_active = False
        car.throttle = 0.0

state_widget.observe(on_state_change, names='value')

# Clean Non-Overlapping Layout
center_box = ipywidgets.VBox([
    prediction_widget,
    state_widget
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

sliders_box = ipywidgets.VBox([
    network_output_slider,
    steering_gain_slider,
    steering_bias_slider,
    steering_value_slider,
    throttle_slider
], layout=Layout(align_items='center'))

display(ipywidgets.VBox([center_box, sliders_box]))


### 4. Advanced Steering Controllers (PID & Stanley Controller with 1D Kalman Filtering)

This section provides two advanced autonomous steering controllers:

1. **Stanley Controller (Recommended - Step 1 Upgrade):**
   * Uses **Heading Error ($\psi$)** estimated from Kalman lateral velocity + **Cross-Track Error ($	ext{CTE} = x$)**.
   * Steering formula: $\delta = \psi + \arctan\left(\frac{k \cdot \text{CTE}}{v + \epsilon}\right) + \text{bias}$.
   * Corrects heading and lateral offset simultaneously without needing data retraining!

2. **PID Controller:**
   * Traditional Proportional, Integral ($K_i$), and Derivative ($K_d$) controller with anti-windup clamping.

3. **1D Kalman Filtering:**
   * Smooths raw neural network predictions and estimates true lateral velocity $v_x = \frac{dx}{dt}$ without noise spikes.

In [ ]:
import cv2
import ipywidgets
import traitlets
import threading
import time
import json
from IPython.display import display
from ipywidgets import Layout
import Controller

# Initialize Controllers from PID.py
pid = Controller.PIDController()
stanley = Controller.StanleyController()

pid_style = {'description_width': '140px'}

# Controller Sliders
k_stanley_slider = ipywidgets.FloatSlider(description='Stanley Gain (k)', min=0.1, max=3.0, value=1.2, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
kp_slider        = ipywidgets.FloatSlider(description='Kp (PID Gain)', min=0.0, max=3.0, value=1.0, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
ki_slider        = ipywidgets.FloatSlider(description='Ki (Integral)', min=0.0, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
kd_slider        = ipywidgets.FloatSlider(description='Kd (Damping)', min=0.0, max=1.0, value=0.15, step=0.02, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
alpha_slider     = ipywidgets.FloatSlider(description='Alpha (Filter)', min=0.1, max=1.0, value=0.7, step=0.05, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
pid_bias_slider  = ipywidgets.FloatSlider(description='Steering Bias', min=-0.5, max=0.5, value=0.0, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)

base_throttle_slider = ipywidgets.FloatSlider(description='Base Throttle', min=0.05, max=0.5, value=0.20, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)
brake_gain_slider    = ipywidgets.FloatSlider(description='Brake Gain', min=0.0, max=0.4, value=0.10, step=0.01, readout=True, readout_format='.2f', layout=Layout(width='400px'), style=pid_style)

pid_steering_disp = ipywidgets.FloatSlider(description='Live Steering', min=-1.0, max=1.0, value=0.0, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=pid_style)
pid_throttle_disp = ipywidgets.FloatSlider(description='Live Throttle', min=0.0, max=0.5, value=0.15, step=0.01, readout=True, readout_format='.2f', disabled=True, layout=Layout(width='400px'), style=pid_style)

pid_state_widget       = ipywidgets.ToggleButtons(options=['Off', 'Stanley Live Drive', 'PID Live Drive'], description='Control Mode', value='Off')
pid_prediction_widget = ipywidgets.Image(value=bgr8_to_jpeg(camera.value), format='jpeg', width=camera.width, height=camera.height)
pid_reset_button       = ipywidgets.Button(description='Reset PID & Stop', button_style='warning', icon='refresh')
load_best_btn          = ipywidgets.Button(description='Load Best Config', button_style='info', icon='download')

pid_live_active = False

def load_best_config(b=None):
    cfg_path = 'best_pid_config.json'
    if os.path.exists(cfg_path):
        with open(cfg_path, 'r', encoding='utf-8') as f:
            cfg = json.load(f)
        kp_slider.value            = cfg.get('kp', kp_slider.value)
        ki_slider.value            = cfg.get('ki', ki_slider.value)
        kd_slider.value            = cfg.get('kd', kd_slider.value)
        alpha_slider.value         = cfg.get('alpha', alpha_slider.value)
        pid_bias_slider.value      = cfg.get('bias', pid_bias_slider.value)
        base_throttle_slider.value = cfg.get('base_throttle', base_throttle_slider.value)
        brake_gain_slider.value    = cfg.get('brake_gain', brake_gain_slider.value)
        print(f"Loaded best config from '{cfg_path}'!")
    else:
        print(f"Config file '{cfg_path}' not found!")

load_best_btn.on_click(load_best_config)

def pid_drive_loop():
    global pid_live_active
    pid.reset()
    stanley.reset()
    while pid_live_active:
        try:
            image = camera.value
            preprocessed = preprocess(image)
            
            with torch.no_grad():
                output = model(preprocessed).detach().cpu().numpy().flatten()
            
            raw_x = float(output[0])
            raw_y = float(output[1]) if len(output) > 1 else 0.0
            
            mode = pid_state_widget.value
            if mode == 'Stanley Live Drive':
                steering, dyn_throttle = stanley.update(
                    raw_x=raw_x,
                    k=k_stanley_slider.value,
                    base_throttle=base_throttle_slider.value,
                    brake_gain=brake_gain_slider.value,
                    bias=pid_bias_slider.value,
                    alpha=alpha_slider.value
                )
                smoothed_x = stanley.smoothed_x
            else:
                steering = pid.update(
                    raw_x=raw_x,
                    kp=kp_slider.value,
                    ki=ki_slider.value,
                    kd=kd_slider.value,
                    alpha=alpha_slider.value,
                    bias=pid_bias_slider.value
                )
                dyn_throttle = base_throttle_slider.value - brake_gain_slider.value * abs(steering)
                dyn_throttle = max(0.05, min(0.5, dyn_throttle))
                smoothed_x = pid.smoothed_x
            
            pid_steering_disp.value = steering
            pid_throttle_disp.value = dyn_throttle
            
            # Command physical JetRacer actuators
            car.steering = steering
            car.throttle = dyn_throttle
            
            # Draw target dot (green circle for target)
            px = int(camera.width  * (smoothed_x / 2.0 + 0.5))
            py = int(camera.height * (raw_y / 2.0 + 0.5))
            
            prediction = image.copy()
            prediction = cv2.circle(prediction, (px, py), 8, (0, 255, 0), 3)
            pid_prediction_widget.value = bgr8_to_jpeg(prediction)
            
        except Exception as e:
            print(f"Error in drive loop: {e}")
            break
            
        time.sleep(0.02)  # ~50 FPS loop

def on_pid_state_change(change):
    global pid_live_active
    if change['new'] in ['PID Live Drive', 'Stanley Live Drive']:
        if not pid_live_active:
            pid_live_active = True
            t = threading.Thread(target=pid_drive_loop, daemon=True)
            t.start()
    else:
        pid_live_active = False
        car.throttle = 0.0

pid_state_widget.observe(on_pid_state_change, names='value')

def on_pid_reset_clicked(b):
    global pid_live_active
    pid_state_widget.value = 'Off'
    pid_live_active = False
    car.throttle = 0.0
    car.steering = 0.0
    time.sleep(0.1)
    pid.reset()
    stanley.reset()
    pid_prediction_widget.value = bgr8_to_jpeg(camera.value)

pid_reset_button.on_click(on_pid_reset_clicked)

# Clean Non-Overlapping Layout
pid_center_box = ipywidgets.VBox([
    pid_prediction_widget,
    ipywidgets.HBox([pid_state_widget, pid_reset_button, load_best_btn])
], layout=Layout(align_items='center', margin='0px 0px 15px 0px'))

pid_tuning_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>Steering Tuning (Stanley & PID)</h4>"),
    k_stanley_slider,
    kp_slider,
    ki_slider,
    kd_slider,
    alpha_slider,
    pid_bias_slider
], layout=Layout(margin='0px 20px 0px 0px'))

throttle_tuning_box = ipywidgets.VBox([
    ipywidgets.HTML(value="<h4>Dynamic Throttle & Outputs</h4>"),
    base_throttle_slider,
    brake_gain_slider,
    ipywidgets.HTML(value="<b>Live Outputs:</b>"),
    pid_steering_disp,
    pid_throttle_disp
])

controls_grid = ipywidgets.HBox([pid_tuning_box, throttle_tuning_box], layout=Layout(justify_content='space-around'))

display(ipywidgets.VBox([pid_center_box, controls_grid]))
